In [ ]:
import sys
import os
import pandas as pd
import polars as pl

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory
from data_preprocessing.repartition import Repartition
from feature_engineering.order_flow import OrderFlowFeatureEngineering

from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline

# Shared configuration
BAR_DURATION_MS = 250

In [ ]:
# Repartition Pipeline with v2 prefix
config_repart = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://orderflowanalysis/intermediate/normalized',
        start_date='2024-05-03',
        end_date='2024-05-03',
        exchanges=['AMERICAS'],
        data_types=['trades']
    ),
    processing=ProcessingConfig(
        repartition=Repartition(partition_column='Ticker', max_retries=3, log_interval=1)
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        repartitioned=S3Location(path='s3://orderflowanalysis/intermediate/repartitioned_v2'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir}, flat_core_count=3)
)

specific_files = ['s3://orderflowanalysis/intermediate/normalized/2023/03/13/trades/AMERICAS/trades-XNAS-20230313.parquet',
                  's3://orderflowanalysis/intermediate/normalized/2023/03/13/level2q/AMERICAS/XNAS-20230313.parquet']

print("Testing Repartition Pipeline (v2 - first letter partitioning)...")
pipeline_repart = Pipeline(config_repart)
results_repart = pipeline_repart.run(specific_files=specific_files)

In [ ]:
# Verify repartition results
if results_repart:
    res_pd = pd.DataFrame(results_repart)
    print(f"Total: {len(results_repart)}, Success: {(res_pd['message']=='success').sum()}, Failed: {(res_pd['message']!='success').sum()}")
    if 'partition_value' in res_pd.columns:
        print(res_pd.groupby(['data_type','partition_value']).size())
    for r in results_repart[:5]:
        print(f"{r['data_type']} [{r.get('partition_value')}]: {r.get('row_count',0):,} rows -> {r['output_path']}")